In [1]:
import json
import os
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
from pydantic_ai import Agent, RunUsage
import tools
from helper import start
from skill import SkillLoader, SkillsTool

In [2]:
# Load environment variables from the .env file
load_dotenv()

api_key = os.environ.get("OPENAI_API_KEY")
openai_client = OpenAI(api_key=api_key)
model = "gpt-6-luna"  # or your preferred model

#old testing prompt
#system_prompt = "You are a helpful assistant that can analyze the project structure and read files to answer questions about the project."


In [3]:
#sytem prompt
django_agent_prompt = """
You are a coding agent. Your task is to modify the provided Django project template
according to user instructions. You don't tell the user what to do; you do it yourself using the
available tools. First, think about the sequence of steps you will do, and then
execute the sequence.
Always ensure changes are consistent with Django best practices and the project's structure.

## Project Overview
The project is a Django 5.2.4 web application scaffolded with standard best practices. It uses:
- Python 3.12+
- Django 5.2.4 (as specified in pyproject.toml)
- uv for Python environment and dependency management
- SQLite as the default database (see settings.py)
- Standard Django apps and a custom app called myapp
- HTML templates for rendering views
- TailwindCSS for styling

## File Tree
""".strip()

In [4]:

skill_loader = SkillLoader(Path("skills/"))
skills_tool = SkillsTool(skill_loader)

SKILL_INJECTION_PROMPT = f'''
You have the following skills available which you can load with the skills tool:

{skill_loader.get_description()}
'''.strip()

AGENT_WITH_SKILLS_INSTRUCTIONS = django_agent_prompt + '\n\n' + SKILL_INJECTION_PROMPT

In [6]:
#creating a new django project
project_name = input("Enter the new Django project name: ").strip()
start(project_name)

Enter the new Django project name:  sudokubg


Django template copied to 'sudokubg' directory.


'sudokubg'

In [8]:
#project location
project_path = Path(project_name)

In [9]:
coding_agent = Agent(
    'openai:gpt-6-luna',
    instructions=AGENT_WITH_SKILLS_INSTRUCTIONS,
    tools=[
        agent_tools.read_file,
        agent_tools.write_file,
        agent_tools.see_file_tree,
        agent_tools.execute_bash_command,
        agent_tools.search_in_files,
        skills_tool.skill,
    ],
)

In [10]:
#To show progress in the notebook, we use a helper that prints each message kind and its content:
def print_messages(messages):
    for m in messages:
        print(m.kind)
        for p in m.parts:
            part_kind = p.part_kind
            if part_kind == "user-prompt":
                print("USER:", p.content)
            if part_kind == "tool-call":
                print("TOOL CALL:", p.tool_name, p.args)
            if part_kind == "tool-return":
                print("TOOL RETURN:", p.tool_name)
            if part_kind == "text":
                print("ASSISTANT:", p.content)
        print()

In [12]:
#The Q&A loop with message_history
#PydanticAI supports message_history so we can keep the conversation history across turns:

messages = []
usage = RunUsage()

while True:
    user_prompt = input(">> ").strip()
    if not user_prompt or user_prompt.lower() == "stop":
        break

    result = await coding_agent.run(
        user_prompt,
        message_history=messages,
    )
    usage += result.usage

    print_messages(result.new_messages())
    messages.extend(result.new_messages())



>>  what is the content of the folder


request
USER: what is the content of the folder

response
TOOL CALL: see_file_tree {}

request
TOOL RETURN: see_file_tree

response
ASSISTANT: The project folder contains:

- **`myapp/`** — Django app, with `admin.py`, `apps.py`, `models.py`, `tests.py`, `views.py`, and a `migrations/` folder. It also contains the template `templates/home.html`.
- **`myproject/`** — Django project configuration: `settings.py`, `urls.py`, `asgi.py`, and `wsgi.py`.
- **`static/css/`** — Static stylesheet: `styles.css`.
- **`templates/`** — Shared template: `base.html`.
- **Root files** — `manage.py`, `pyproject.toml`, `uv.lock`, `README.md`, `Makefile`, `.gitignore`, and `db.sqlite3`.



>>  stop
